In [4]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load the cleaned data
df = pd.read_csv('../data/processed/cleaned_telco_churn.csv')

print("✅ Cleaned data loaded successfully!")
print(f"Dataset Shape: {df.shape}")
print("\nFirst few rows:")
df.head()

✅ Cleaned data loaded successfully!
Dataset Shape: (7043, 21)

First few rows:


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [5]:
# Create tenure-based features
print("CREATING TENURE-BASED FEATURES")
print("="*50)

# 1. Tenure groups (categorical)
df['tenure_group'] = pd.cut(df['tenure'], 
                            bins=[0, 12, 24, 48, 72], 
                            labels=['0-1 Year', '1-2 Years', '2-4 Years', '4+ Years'])

# 2. Is new customer (binary)
df['is_new_customer'] = (df['tenure'] <= 12).astype(int)

# 3. Tenure in years (continuous)
df['tenure_years'] = df['tenure'] / 12

# 4. Customer lifecycle stage
def get_lifecycle_stage(tenure):
    if tenure <= 6:
        return 'New'
    elif tenure <= 24:
        return 'Growing'
    elif tenure <= 48:
        return 'Mature'
    else:
        return 'Loyal'

df['lifecycle_stage'] = df['tenure'].apply(get_lifecycle_stage)

print("✅ Created 4 tenure-based features:")
print("   - tenure_group")
print("   - is_new_customer")
print("   - tenure_years")
print("   - lifecycle_stage")

print(f"\nNew dataset shape: {df.shape}")
print("\nSample of new features:")
df[['customerID', 'tenure', 'tenure_group', 'is_new_customer', 'tenure_years', 'lifecycle_stage']].head(10)

CREATING TENURE-BASED FEATURES
✅ Created 4 tenure-based features:
   - tenure_group
   - is_new_customer
   - tenure_years
   - lifecycle_stage

New dataset shape: (7043, 25)

Sample of new features:


,customerID,tenure,tenure_group,is_new_customer,tenure_years,lifecycle_stage
0,7590-VHVEG,1,0-1 Year,1,0.083333,New
1,5575-GNVDE,34,2-4 Years,0,2.833333,Mature
2,3668-QPYBK,2,0-1 Year,1,0.166667,New
3,7795-CFOCW,45,2-4 Years,0,3.750000,Mature
4,9237-HQITU,2,0-1 Year,1,0.166667,New
5,9305-CDSKC,8,0-1 Year,1,0.666667,Growing
6,1452-KIOVK,22,1-2 Years,0,1.833333,Growing
7,6713-OKOMC,10,0-1 Year,1,0.833333,Growing
8,7892-POOKP,28,2-4 Years,0,2.333333,Mature
9,6388-TABGU,62,4+ Years,0,5.166667,Loyal


In [6]:
# Create revenue-based features
print("CREATING REVENUE-BASED FEATURES")
print("="*50)

# 1. Average monthly spending (TotalCharges / tenure)
# Handle division by zero for tenure = 0
df['avg_monthly_spending'] = df.apply(
    lambda row: row['MonthlyCharges'] if row['tenure'] == 0 
    else row['TotalCharges'] / row['tenure'], 
    axis=1
)

# 2. Price tier (categorize customers by monthly charges)
df['price_tier'] = pd.cut(df['MonthlyCharges'], 
                          bins=[0, 35, 70, 120], 
                          labels=['Low', 'Medium', 'High'])

# 3. Spending trend (is total charges higher than expected?)
df['expected_total'] = df['MonthlyCharges'] * df['tenure']
df['spending_ratio'] = df['TotalCharges'] / (df['expected_total'] + 1)  # +1 to avoid division by zero

# 4. High value customer (top 25% by total charges)
top_25_percentile = df['TotalCharges'].quantile(0.75)
df['is_high_value'] = (df['TotalCharges'] >= top_25_percentile).astype(int)

print("✅ Created 4 revenue-based features:")
print("   - avg_monthly_spending")
print("   - price_tier")
print("   - spending_ratio")
print("   - is_high_value")

print(f"\nNew dataset shape: {df.shape}")
print("\nSample of revenue features:")
df[['customerID', 'MonthlyCharges', 'TotalCharges', 'avg_monthly_spending', 'price_tier', 'is_high_value']].head(10)

CREATING REVENUE-BASED FEATURES
✅ Created 4 revenue-based features:
   - avg_monthly_spending
   - price_tier
   - spending_ratio
   - is_high_value

New dataset shape: (7043, 30)

Sample of revenue features:


,customerID,MonthlyCharges,TotalCharges,avg_monthly_spending,price_tier,is_high_value
0,7590-VHVEG,29.85,29.85,29.850000,Low,0
1,5575-GNVDE,56.95,1889.50,55.573529,Medium,0
2,3668-QPYBK,53.85,108.15,54.075000,Medium,0
3,7795-CFOCW,42.30,1840.75,40.905556,Medium,0
4,9237-HQITU,70.70,151.65,75.825000,High,0
5,9305-CDSKC,99.65,820.50,102.562500,High,0
6,1452-KIOVK,89.10,1949.40,88.609091,High,0
7,6713-OKOMC,29.75,301.90,30.190000,Low,0
8,7892-POOKP,104.80,3046.05,108.787500,High,0
9,6388-TABGU,56.15,3487.95,56.257258,Medium,0


In [7]:
# Create service usage features
print("CREATING SERVICE USAGE FEATURES")
print("="*50)

# 1. Count total services used
service_cols = ['PhoneService', 'InternetService', 'OnlineSecurity', 
                'OnlineBackup', 'DeviceProtection', 'TechSupport', 
                'StreamingTV', 'StreamingMovies']

# Function to count services
def count_services(row):
    count = 0
    if row['PhoneService'] == 'Yes':
        count += 1
    if row['InternetService'] != 'No':
        count += 1
    if row['OnlineSecurity'] == 'Yes':
        count += 1
    if row['OnlineBackup'] == 'Yes':
        count += 1
    if row['DeviceProtection'] == 'Yes':
        count += 1
    if row['TechSupport'] == 'Yes':
        count += 1
    if row['StreamingTV'] == 'Yes':
        count += 1
    if row['StreamingMovies'] == 'Yes':
        count += 1
    return count

df['total_services'] = df.apply(count_services, axis=1)

# 2. Has internet service (binary)
df['has_internet'] = (df['InternetService'] != 'No').astype(int)

# 3. Has phone service (binary)
df['has_phone'] = (df['PhoneService'] == 'Yes').astype(int)

# 4. Has streaming services (either TV or Movies)
df['has_streaming'] = ((df['StreamingTV'] == 'Yes') | (df['StreamingMovies'] == 'Yes')).astype(int)

# 5. Has security services (either OnlineSecurity or DeviceProtection)
df['has_security'] = ((df['OnlineSecurity'] == 'Yes') | (df['DeviceProtection'] == 'Yes')).astype(int)

# 6. Service engagement level
def get_engagement_level(total_services):
    if total_services <= 2:
        return 'Low'
    elif total_services <= 4:
        return 'Medium'
    else:
        return 'High'

df['engagement_level'] = df['total_services'].apply(get_engagement_level)

print("✅ Created 6 service-based features:")
print("   - total_services")
print("   - has_internet")
print("   - has_phone")
print("   - has_streaming")
print("   - has_security")
print("   - engagement_level")

print(f"\nNew dataset shape: {df.shape}")
print("\nService usage distribution:")
print(df['total_services'].value_counts().sort_index())
print("\nEngagement level distribution:")
print(df['engagement_level'].value_counts())

CREATING SERVICE USAGE FEATURES
✅ Created 6 service-based features:
   - total_services
   - has_internet
   - has_phone
   - has_streaming
   - has_security
   - engagement_level

New dataset shape: (7043, 36)

Service usage distribution:
total_services
1    1606
2     727
3     996
4    1041
5    1062
6     827
7     525
8     259
Name: count, dtype: int64

Engagement level distribution:
engagement_level
High      2673
Low       2333
Medium    2037
Name: count, dtype: int64


In [8]:
# Create contract and billing features
print("CREATING CONTRACT & BILLING FEATURES")
print("="*50)

# 1. Contract commitment level (numeric encoding)
contract_mapping = {'Month-to-month': 0, 'One year': 1, 'Two year': 2}
df['contract_level'] = df['Contract'].map(contract_mapping)

# 2. Is on long-term contract (binary)
df['has_long_contract'] = (df['Contract'] != 'Month-to-month').astype(int)

# 3. Uses paperless billing (already in data, convert to binary)
df['uses_paperless'] = (df['PaperlessBilling'] == 'Yes').astype(int)

# 4. Payment method risk (some payment methods have higher churn)
# Electronic check has been shown to have higher churn
df['risky_payment'] = (df['PaymentMethod'] == 'Electronic check').astype(int)

# 5. Auto payment (binary - if using automatic payment methods)
auto_payment_methods = ['Bank transfer (automatic)', 'Credit card (automatic)']
df['uses_auto_payment'] = df['PaymentMethod'].isin(auto_payment_methods).astype(int)

print("✅ Created 5 contract/billing features:")
print("   - contract_level")
print("   - has_long_contract")
print("   - uses_paperless")
print("   - risky_payment")
print("   - uses_auto_payment")

print(f"\nNew dataset shape: {df.shape}")
print("\nContract distribution:")
print(df['Contract'].value_counts())
print("\nPayment method distribution:")
print(df['PaymentMethod'].value_counts())

CREATING CONTRACT & BILLING FEATURES
✅ Created 5 contract/billing features:
   - contract_level
   - has_long_contract
   - uses_paperless
   - risky_payment
   - uses_auto_payment

New dataset shape: (7043, 41)

Contract distribution:
Contract
Month-to-month    3875
Two year          1695
One year          1473
Name: count, dtype: int64

Payment method distribution:
PaymentMethod
Electronic check             2365
Mailed check                 1612
Bank transfer (automatic)    1544
Credit card (automatic)      1522
Name: count, dtype: int64


In [9]:
# Create demographic features
print("CREATING DEMOGRAPHIC FEATURES")
print("="*50)

# 1. Has family (has partner OR dependents)
df['has_family'] = ((df['Partner'] == 'Yes') | (df['Dependents'] == 'Yes')).astype(int)

# 2. Family size (numeric representation)
def get_family_size(row):
    size = 1  # The customer themselves
    if row['Partner'] == 'Yes':
        size += 1
    if row['Dependents'] == 'Yes':
        size += 1  # Simplified - assuming at least 1 dependent
    return size

df['family_size'] = df.apply(get_family_size, axis=1)

# 3. Senior with dependents (specific segment)
df['senior_with_dependents'] = ((df['SeniorCitizen'] == 1) & (df['Dependents'] == 'Yes')).astype(int)

# 4. Single senior (higher risk segment)
df['single_senior'] = ((df['SeniorCitizen'] == 1) & (df['Partner'] == 'No') & (df['Dependents'] == 'No')).astype(int)

# 5. Customer profile category
def get_customer_profile(row):
    if row['SeniorCitizen'] == 1:
        if row['has_family'] == 1:
            return 'Senior with Family'
        else:
            return 'Senior Alone'
    else:
        if row['has_family'] == 1:
            return 'Adult with Family'
        else:
            return 'Adult Alone'

df['customer_profile'] = df.apply(get_customer_profile, axis=1)

print("✅ Created 5 demographic features:")
print("   - has_family")
print("   - family_size")
print("   - senior_with_dependents")
print("   - single_senior")
print("   - customer_profile")

print(f"\nNew dataset shape: {df.shape}")
print("\nCustomer profile distribution:")
print(df['customer_profile'].value_counts())
print("\nFamily size distribution:")
print(df['family_size'].value_counts())

CREATING DEMOGRAPHIC FEATURES
✅ Created 5 demographic features:
   - has_family
   - family_size
   - senior_with_dependents
   - single_senior
   - customer_profile

New dataset shape: (7043, 46)

Customer profile distribution:
customer_profile
Adult with Family     3182
Adult Alone           2719
Senior with Family     581
Senior Alone           561
Name: count, dtype: int64

Family size distribution:
family_size
1    3280
2    2014
3    1749
Name: count, dtype: int64


In [10]:
# Create advanced risk features
print("CREATING ADVANCED RISK FEATURES")
print("="*50)

# 1. Churn risk score (based on patterns we discovered)
def calculate_risk_score(row):
    risk = 0
    
    # Tenure risk
    if row['tenure'] <= 12:
        risk += 3  # New customers are highest risk
    elif row['tenure'] <= 24:
        risk += 2
    elif row['tenure'] <= 48:
        risk += 1
    
    # Contract risk
    if row['Contract'] == 'Month-to-month':
        risk += 3  # Month-to-month is high risk
    
    # Payment risk
    if row['PaymentMethod'] == 'Electronic check':
        risk += 2
    
    # Service engagement risk
    if row['total_services'] <= 2:
        risk += 1  # Low engagement
    
    # Price risk
    if row['MonthlyCharges'] > 80:
        risk += 1  # High price sensitivity
    
    return risk

df['churn_risk_score'] = df.apply(calculate_risk_score, axis=1)

# 2. High risk customer (binary)
df['is_high_risk'] = (df['churn_risk_score'] >= 6).astype(int)

# 3. Value-to-risk ratio
df['value_risk_ratio'] = df['TotalCharges'] / (df['churn_risk_score'] + 1)

# 4. Engagement per dollar (services per monthly charge)
df['engagement_per_dollar'] = df['total_services'] / df['MonthlyCharges']

print("✅ Created 4 advanced risk features:")
print("   - churn_risk_score")
print("   - is_high_risk")
print("   - value_risk_ratio")
print("   - engagement_per_dollar")

print(f"\nNew dataset shape: {df.shape}")
print("\nRisk score distribution:")
print(df['churn_risk_score'].value_counts().sort_index())
print(f"\nHigh risk customers: {df['is_high_risk'].sum()} ({df['is_high_risk'].sum()/len(df)*100:.1f}%)")

CREATING ADVANCED RISK FEATURES
✅ Created 4 advanced risk features:
   - churn_risk_score
   - is_high_risk
   - value_risk_ratio
   - engagement_per_dollar

New dataset shape: (7043, 50)

Risk score distribution:
churn_risk_score
0     438
1    1365
2     566
3     575
4     512
5     478
6     764
7    1130
8     535
9     680
Name: count, dtype: int64

High risk customers: 3109 (44.1%)


In [11]:
# Summary of all features created
print("FEATURE ENGINEERING SUMMARY")
print("="*60)

print(f"\n📊 Original features: 21")
print(f"📊 New features created: {df.shape[1] - 21}")
print(f"📊 Total features now: {df.shape[1]}")

print("\n✅ Feature Categories Created:")
print("   1. Tenure-based (4): tenure_group, is_new_customer, tenure_years, lifecycle_stage")
print("   2. Revenue-based (4): avg_monthly_spending, price_tier, spending_ratio, is_high_value")
print("   3. Service usage (6): total_services, has_internet, has_phone, has_streaming, has_security, engagement_level")
print("   4. Contract/Billing (5): contract_level, has_long_contract, uses_paperless, risky_payment, uses_auto_payment")
print("   5. Demographic (5): has_family, family_size, senior_with_dependents, single_senior, customer_profile")
print("   6. Advanced Risk (4): churn_risk_score, is_high_risk, value_risk_ratio, engagement_per_dollar")

print("\n" + "="*60)

# Display all column names
print("\nAll features in dataset:")
for i, col in enumerate(df.columns, 1):
    print(f"   {i}. {col}")

print("\n" + "="*60)

# Save the feature-engineered dataset
df.to_csv('../data/processed/telco_churn_with_features.csv', index=False)

print("\n✅ Feature-engineered dataset saved!")
print("   Location: data/processed/telco_churn_with_features.csv")
print(f"   Rows: {df.shape[0]}")
print(f"   Columns: {df.shape[1]}")

FEATURE ENGINEERING SUMMARY

📊 Original features: 21
📊 New features created: 29
📊 Total features now: 50

✅ Feature Categories Created:
   1. Tenure-based (4): tenure_group, is_new_customer, tenure_years, lifecycle_stage
   2. Revenue-based (4): avg_monthly_spending, price_tier, spending_ratio, is_high_value
   3. Service usage (6): total_services, has_internet, has_phone, has_streaming, has_security, engagement_level
   4. Contract/Billing (5): contract_level, has_long_contract, uses_paperless, risky_payment, uses_auto_payment
   5. Demographic (5): has_family, family_size, senior_with_dependents, single_senior, customer_profile
   6. Advanced Risk (4): churn_risk_score, is_high_risk, value_risk_ratio, engagement_per_dollar


All features in dataset:
   1. customerID
   2. gender
   3. SeniorCitizen
   4. Partner
   5. Dependents
   6. tenure
   7. PhoneService
   8. MultipleLines
   9. InternetService
   10. OnlineSecurity
   11. OnlineBackup
   12. DeviceProtection
   13. TechSuppor